In [ ]:
%py
# PySpark script to mask the last 4 digits of the invoice_number in d_product_revenue_clone and handle various test scenarios

from pyspark.sql.functions import col, expr, lit, when
from pyspark.sql.types import StructType, StructField, StringType, LongType, DateType, DoubleType

# Define the schema for the table
schema = StructType([
    StructField("product_id", LongType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_type", StringType(), True),
    StructField("revenue", LongType(), True),
    StructField("country", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("purchased_date", DateType(), True),
    StructField("invoice_date", DateType(), True),
    StructField("invoice_number", LongType(), True),
    StructField("is_returned", LongType(), True),
    StructField("customer_satisfaction_score", LongType(), True),
    StructField("product_details", StringType(), True),
    StructField("customer_first_purchased_date", DateType(), True),
    StructField("customer_first_product", StringType(), True),
    StructField("customer_first_revenue", DoubleType(), True)
])

# 1. Drop the existing d_product_revenue_clone table if it exists
spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone")

# 2. Create a clone of the d_product_revenue table
spark.sql("""
CREATE TABLE purgo_playground.d_product_revenue_clone
AS SELECT * FROM purgo_playground.d_product_revenue
""")

# 3. Load the table into a DataFrame
df_clone = spark.table("purgo_playground.d_product_revenue_clone")

# 4. Apply the masking logic to the invoice_number
df_masked = df_clone.withColumn(
    "masked_invoice_number",
    when(col("invoice_number").isNull(), lit(None))  # Handle NULL scenarios by leaving them as NULL
    .otherwise(
        expr("CAST(invoice_number AS STRING)")  # Convert invoice_number to STRING for manipulation
        .rlike("^[0-9]{10}$")  # Validate that it's a 10-digit number
        .cast(StringType())
        .substr(1, 6).concat(lit("****"))  # Mask the last 4 digits
    )
)

# 5. Display the masked DataFrame for verification (The print statement illustrates the output, replace with actual save or display command in appropriate context)
df_masked.select("invoice_number", "masked_invoice_number").show(truncate=False)

# 6. Update the table with the masked values
df_masked.select(
    "product_id", "product_name", "product_type", "revenue", "country",
    "customer_id", "purchased_date", "invoice_date", col("masked_invoice_number").alias("invoice_number"),
    "is_returned", "customer_satisfaction_score", "product_details",
    "customer_first_purchased_date", "customer_first_product", "customer_first_revenue"
).write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")

# Note: Consider appropriate exception handling and logging mechanisms in actual production code.

